# 02 — Uni-Mol + XGBoost + SHAP

**Goal:** replace the ECFP4 features with Uni-Mol 3D embeddings, retrain
XGBoost, and explain predictions with TreeSHAP.

Read `docs/BACKGROUND.md` §3 Gen 4 (Uni-Mol) and §3 Gen 5 sub-thread A
(SHAP) before running. Heavy step: first call downloads Uni-Mol weights
from HuggingFace.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
from qsar_tutorial.data import load_coadd_pa, stratified_folds, Dataset
from qsar_tutorial.featurizer import UniMolFeaturizer
from qsar_tutorial.model import build_classifier, cross_validated_oof, fit_with_class_weight
from qsar_tutorial.shap_layer import explain

ds = load_coadd_pa('../data/processed/coadd_pa.csv')
print(len(ds), ds.active_fraction)

### Extract Uni-Mol embeddings (slow on first run)

In [ ]:
feat = UniMolFeaturizer(model_name='unimolv1', model_size='84m')
X, valid = feat.featurize(list(ds.smiles))
X, y = X[valid], ds.y[valid]
smis = ds.smiles[valid]
print(X.shape)

### OOF performance and TreeSHAP

In [ ]:
folds = stratified_folds(Dataset(smiles=smis, y=y), n_splits=5)
oof = cross_validated_oof(X, y, folds, n_estimators=300, max_depth=6)
print(oof.summary())

final = build_classifier(n_estimators=300, max_depth=6)
fit_with_class_weight(final, X, y)
sv = explain(final, X, feature_names=[f'unimol_{i}' for i in range(X.shape[1])])
for name, imp in sv.top_features(10):
    print(f'{name:>14}  {imp:.4f}')

### Reading SHAP attributions on a foundation-model embedding

Uni-Mol dimensions are not chemically named. SHAP tells you 'the model
relies on dim 273'. To turn that into chemistry you need either:

- **Notebook 03 → SAE**: decompose the 768-d embedding into ~6000
  sparser features, many of which align with RDKit descriptors.
- **Per-atom attribution** (advanced): trace SHAP weights back to
  atomic representations via the Uni-Mol atomic-repr API — see the
  parent project for the full implementation.